# Public Data Case Studies

**Audience:** practitioners auditing public mining datasets.  
**Goals:** keep the three case studies independent, inspect local availability, train a temporally
valid soft-sensor baseline, and understand the GeoMet spatial holdout contract.

Run `python -m mineral_process.cli download-data` before this notebook to use the real files.

In [1]:
import sys
from pathlib import Path

ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src" / "mineral_process").exists()
)
sys.path.insert(0, str(ROOT / "src"))

## 1. Inventory: never concatenate these plants

In [2]:
from pathlib import Path

cases = {
    "GeoMet": ROOT / "data/raw/geomet",
    "Iron flotation": ROOT / "data/raw/iron_flotation",
    "Polymetallic": ROOT / "data/raw/polymetallic",
}
{name: [p.name for p in path.glob("*") if p.name != ".gitkeep"] for name, path in cases.items()}

{'GeoMet': ['drillholes.csv', 'comminution.csv', 'flotation.csv'],
 'Iron flotation': ['quality-prediction-in-a-mining-process.zip',
  'MiningProcess_Flotation_Plant_Database.csv'],
 'Polymetallic': ['Base de datos.xlsx']}

## 2. Temporal soft sensor with a deterministic fallback

In [3]:
from mineral_process.data.loaders import load_iron_flotation
from mineral_process.models.soft_sensor import (
    prepare_temporal_soft_sensor_frame,
    synthetic_flotation_data,
    train_soft_sensor,
)

try:
    # A chronological prefix keeps this tutorial interactive on synced drives.
    # Use nrows=None in production training to load the complete dataset.
    flotation = load_iron_flotation(ROOT / "data/raw/iron_flotation", nrows=120_000)
    data_origin = "real public Kaggle case study"
except FileNotFoundError:
    flotation = synthetic_flotation_data(rows=2400, seed=42)
    data_origin = "synthetic fallback"
data_origin, flotation.shape

('real public Kaggle case study', (120000, 24))

In [4]:
baseline_bundle = train_soft_sensor(flotation)
temporal_frame, target = prepare_temporal_soft_sensor_frame(flotation)
bundle = train_soft_sensor(temporal_frame, target)
{"baseline": baseline_bundle.metrics, "causal_temporal": bundle.metrics}

{'baseline': ModelMetrics(mae=0.8526455044417283, rmse=1.075059887438191, r2=-0.59976902945421, rows_train=84000, rows_test=18000),
 'causal_temporal': ModelMetrics(mae=0.8856783311857747, rmse=1.084612274732192, r2=-0.6141402181795219, rows_train=466, rows_test=101)}

The feature list must not contain `% Iron Concentrate`: it generally comes from the same delayed lab assay as the target.

In [5]:
assert all("iron concentrate" not in feature.lower() for feature in bundle.features)
bundle.predict(temporal_frame.tail(5))

,prediction,lower,upper,model_version
date,,,,
2017-04-19 21:00:00+00:00,1.879662,-0.182826,3.942149,soft-sensor-v0.1.0
2017-04-19 22:00:00+00:00,1.617695,-0.444793,3.680183,soft-sensor-v0.1.0
2017-04-19 23:00:00+00:00,1.552597,-0.509890,3.615085,soft-sensor-v0.1.0
2017-04-20 00:00:00+00:00,1.429295,-0.633193,3.491782,soft-sensor-v0.1.0
2017-04-20 01:00:00+00:00,1.407847,-0.654641,3.470335,soft-sensor-v0.1.0


## 3. GeoMet spatial-block contract

In [6]:
import pandas as pd

from mineral_process.geomet.spatial import spatial_block_ids

grid = pd.DataFrame({"x": range(16), "y": [v % 4 for v in range(16)]})
grid["block"] = spatial_block_ids(grid.x, grid.y, blocks_per_axis=4)
grid

,x,y,block
0,0,0,0
1,1,1,1
2,2,2,2
3,3,3,3
4,4,0,4
5,5,1,5
6,6,2,6
7,7,3,7
8,8,0,8
9,9,1,9


## Exercise

Inspect `bundle.metrics`. Compare it with a mean-only baseline on the final 15% of rows. Do not
shuffle the observations. The answer scaffold deliberately leaves the final calculation to you.

In [7]:
# Answer scaffold
target = bundle.target
cut = int(len(flotation) * 0.85)
training_mean = flotation.iloc[:cut][target].mean()
holdout = flotation.iloc[cut:][target].dropna()
baseline_mae = (holdout - training_mean).abs().mean()
{"baseline_mae": baseline_mae, "model_mae": bundle.metrics.mae}

{'baseline_mae': np.float64(0.7096226104999811),
 'model_mae': 0.8856783311857747}

## Pitfall and extension

**Pitfall:** a random split overstates industrial performance because adjacent timestamps or spatial
neighbors leak regimes. **Extension:** implement rolling-origin evaluation and compare metric
dispersion across time windows or spatial blocks.